### Introduction
A junior data analyst working on the marketing analyst team at Cyclistic, a bike-share 
company in Chicago. The director of marketing believes the company’s future success 
depends on maximizing the number of annual memberships. Therefore, your team wants to 
understand how casual riders and annual members use Cyclistic bikes differently. From these 
insights, your team will design a new marketing strategy to convert casual riders into annual 
members.

### Business task
 1. How do annual members and casual riders use Cyclistic bikes differently?

### Prepare data
### Download 2019 and 2020 Q1 link
 (https://divvy-tripdata.s3.amazonaws.com/index.html).

In [1]:
# import libraries
import numpy as np
import pandas as pd
# Load Divvy datasets csv files 
df1 = pd.read_csv("Divvy_Trips_2019_Q1.csv")
df2 = pd.read_csv("Divvy_Trips_2020_Q1.csv")
# check and review df


df1.head()

,trip_id,start_time,end_time,bikeid,tripduration,from_station_id,from_station_name,to_station_id,to_station_name,usertype,gender,birthyear
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,2167,390.0,199,Wabash Ave & Grand Ave,84,Milwaukee Ave & Grand Ave,Subscriber,Male,1989.0
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,4386,441.0,44,State St & Randolph St,624,Dearborn St & Van Buren St (*),Subscriber,Female,1990.0
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,1524,829.0,15,Racine Ave & 18th St,644,Western Ave & Fillmore St (*),Subscriber,Female,1994.0
3,21742446,2019-01-01 00:13:45,2019-01-01 00:43:28,252,"1,783.0",123,California Ave & Milwaukee Ave,176,Clark St & Elm St,Subscriber,Male,1993.0
4,21742447,2019-01-01 00:14:52,2019-01-01 00:20:56,1170,364.0,173,Mies van der Rohe Way & Chicago Ave,35,Streeter Dr & Grand Ave,Subscriber,Male,1994.0


In [2]:
df2.head() # check df2

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,EACB19130B0CDA4A,docked_bike,2020-01-21 20:06:59,2020-01-21 20:14:30,Western Ave & Leland Ave,239,Clark St & Leland Ave,326.0,41.9665,-87.6884,41.9671,-87.6674,member
1,8FED874C809DC021,docked_bike,2020-01-30 14:22:39,2020-01-30 14:26:22,Clark St & Montrose Ave,234,Southport Ave & Irving Park Rd,318.0,41.9616,-87.6660,41.9542,-87.6644,member
2,789F3C21E472CA96,docked_bike,2020-01-09 19:29:26,2020-01-09 19:32:17,Broadway & Belmont Ave,296,Wilton Ave & Belmont Ave,117.0,41.9401,-87.6455,41.9402,-87.6530,member
3,C9A388DAC6ABF313,docked_bike,2020-01-06 16:17:07,2020-01-06 16:25:56,Clark St & Randolph St,51,Fairbanks Ct & Grand Ave,24.0,41.8846,-87.6319,41.8918,-87.6206,member
4,943BC3CBECCFD662,docked_bike,2020-01-30 08:37:16,2020-01-30 08:42:48,Clinton St & Lake St,66,Wells St & Hubbard St,212.0,41.8856,-87.6418,41.8899,-87.6343,member



### WRANGLE DATA AND COMBINE INTO A SINGLE FILE (PROCESS PHASE)


In [3]:
# Compare column names for each of the files
print(df1.columns)
print("")
print(df2.columns)

Index(['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration',
       'from_station_id', 'from_station_name', 'to_station_id',
       'to_station_name', 'usertype', 'gender', 'birthyear'],
      dtype='str')

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')


In [4]:
# Rename columns to make them consistent with df2 (2020_Q1)
df1 = df1.rename(columns={
'trip_id': 'ride_id',
'bikeid': 'rideable_type',
'start_time': 'started_at',
'end_time': 'ended_at',
'from_station_name': 'start_station_name',
'from_station_id': 'start_station_id',
'to_station_name': 'end_station_name',
'to_station_id': 'end_station_id',
'usertype': 'member_casual'
})

In [5]:
# Inspect the data frames and look for incongruities
print(df1.info())
print()
print(df2.info())

<class 'pandas.DataFrame'>
RangeIndex: 365069 entries, 0 to 365068
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             365069 non-null  int64  
 1   started_at          365069 non-null  str    
 2   ended_at            365069 non-null  str    
 3   rideable_type       365069 non-null  int64  
 4   tripduration        365069 non-null  str    
 5   start_station_id    365069 non-null  int64  
 6   start_station_name  365069 non-null  str    
 7   end_station_id      365069 non-null  int64  
 8   end_station_name    365069 non-null  str    
 9   member_casual       365069 non-null  str    
 10  gender              345358 non-null  str    
 11  birthyear           347046 non-null  float64
dtypes: float64(1), int64(4), str(7)
memory usage: 33.4 MB
None

<class 'pandas.DataFrame'>
RangeIndex: 426887 entries, 0 to 426886
Data columns (total 13 columns):
 #   Column              Non-Null C

In [6]:
#df1 rider_id and rideable_type are in int64
# Convert ride_id and rideable_type to string so that they can stack correctly with df2

df1['ride_id'] = df1['ride_id'].astype(str)
df1['rideable_type'] = df1['rideable_type'].astype(str)

df1.info() #inspect

<class 'pandas.DataFrame'>
RangeIndex: 365069 entries, 0 to 365068
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             365069 non-null  str    
 1   started_at          365069 non-null  str    
 2   ended_at            365069 non-null  str    
 3   rideable_type       365069 non-null  str    
 4   tripduration        365069 non-null  str    
 5   start_station_id    365069 non-null  int64  
 6   start_station_name  365069 non-null  str    
 7   end_station_id      365069 non-null  int64  
 8   end_station_name    365069 non-null  str    
 9   member_casual       365069 non-null  str    
 10  gender              345358 non-null  str    
 11  birthyear           347046 non-null  float64
dtypes: float64(1), int64(2), str(9)
memory usage: 33.4 MB


In [7]:
# Stack df1 and df2 into 1 dataframe
all_trips = pd.concat([df1,df2],ignore_index=True)
all_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 791956 entries, 0 to 791955
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             791956 non-null  str    
 1   started_at          791956 non-null  str    
 2   ended_at            791956 non-null  str    
 3   rideable_type       791956 non-null  str    
 4   tripduration        365069 non-null  str    
 5   start_station_id    791956 non-null  int64  
 6   start_station_name  791956 non-null  str    
 7   end_station_id      791955 non-null  float64
 8   end_station_name    791955 non-null  str    
 9   member_casual       791956 non-null  str    
 10  gender              345358 non-null  str    
 11  birthyear           347046 non-null  float64
 12  start_lat           426887 non-null  float64
 13  start_lng           426887 non-null  float64
 14  end_lat             426886 non-null  float64
 15  end_lng             426886 non-null  float64


In [8]:
# Remove lat, long, birthyear, and gender fields as this data was not included in 2020_Q1 datasetb

all_trips = all_trips.drop(columns = ['start_lat','start_lng','end_lat','end_lng','gender','birthyear','tripduration'],errors ='ignore')
#'errors="ignore"' at the end prevents an error if a column is not found.

all_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 791956 entries, 0 to 791955
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             791956 non-null  str    
 1   started_at          791956 non-null  str    
 2   ended_at            791956 non-null  str    
 3   rideable_type       791956 non-null  str    
 4   start_station_id    791956 non-null  int64  
 5   start_station_name  791956 non-null  str    
 6   end_station_id      791955 non-null  float64
 7   end_station_name    791955 non-null  str    
 8   member_casual       791956 non-null  str    
dtypes: float64(1), int64(1), str(7)
memory usage: 54.4 MB


### data cleaning and data manipulation

In [9]:
# Inspect the new table that has been created
print("\nList of column names:\n", all_trips.columns)
print("\nHow many rows are in data frame?", len(all_trips))
print("\nDimensions of the data frame:", all_trips.shape)

# Statistical summary of data 
print("\nStatistical summary of data:\n", all_trips.describe())

all_trips.head() # display 1st 5(five) rows


List of column names:
 Index(['ride_id', 'started_at', 'ended_at', 'rideable_type',
       'start_station_id', 'start_station_name', 'end_station_id',
       'end_station_name', 'member_casual'],
      dtype='str')

How many rows are in data frame? 791956

Dimensions of the data frame: (791956, 9)

Statistical summary of data:
        start_station_id  end_station_id
count     791956.000000   791955.000000
mean         204.400294      204.379354
std          158.919132      159.323552
min            2.000000        2.000000
25%           77.000000       77.000000
50%          174.000000      174.000000
75%          291.000000      291.000000
max          675.000000      675.000000


,ride_id,started_at,ended_at,rideable_type,start_station_id,start_station_name,end_station_id,end_station_name,member_casual
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,2167,199,Wabash Ave & Grand Ave,84.0,Milwaukee Ave & Grand Ave,Subscriber
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,4386,44,State St & Randolph St,624.0,Dearborn St & Van Buren St (*),Subscriber
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,1524,15,Racine Ave & 18th St,644.0,Western Ave & Fillmore St (*),Subscriber
3,21742446,2019-01-01 00:13:45,2019-01-01 00:43:28,252,123,California Ave & Milwaukee Ave,176.0,Clark St & Elm St,Subscriber
4,21742447,2019-01-01 00:14:52,2019-01-01 00:20:56,1170,173,Mies van der Rohe Way & Chicago Ave,35.0,Streeter Dr & Grand Ave,Subscriber


### Problems need to fix

In [10]:
""" (1) In the "member_casual" column, there are two names for members ("member" and "Subscriber") and 
two names for casual riders ("Customer" and "casual"). You will need to consolidate that from four to two labels. """

# In the "member_casual" column, replace "Subscriber" with "member" and "Customer" with "casual"

# Begin by discovering how many observations fall under each usertype

print("\nValue counts for 'member_casual' column :\n",all_trips['member_casual'].value_counts())


Value counts for 'member_casual' column :
 member_casual
member        378407
Subscriber    341906
casual         48480
Customer       23163
Name: count, dtype: int64


In [11]:
# Replace the desired values (you can go with the current 2020 labels)
all_trips['member_casual'] = all_trips['member_casual'].replace({
    'Subscriber' : 'member',
    'Customer' : 'casual' })

#inspect again
print("\nValue counts for 'member_casual' column after cleaning:\n",all_trips['member_casual'].value_counts())


Value counts for 'member_casual' column after cleaning:
 member_casual
member    720313
casual     71643
Name: count, dtype: int64


In [12]:
"""(2) The data can only be aggregated at the ride-level, which is too granular. You will want to
add some additional columns of data -- such as day, month, year -- that provide additional
opportunities to aggregate the data. """

# Convert 'started_at' and 'ended_at' to datetime objects
all_trips['started_at'] = pd.to_datetime(all_trips['started_at'])
all_trips['ended_at'] = pd.to_datetime(all_trips['ended_at'])

print(all_trips['started_at'].info())
print()
print(all_trips['ended_at'].info())

<class 'pandas.Series'>
RangeIndex: 791956 entries, 0 to 791955
Series name: started_at
Non-Null Count   Dtype         
--------------   -----         
791956 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 6.0 MB
None

<class 'pandas.Series'>
RangeIndex: 791956 entries, 0 to 791955
Series name: ended_at
Non-Null Count   Dtype         
--------------   -----         
791956 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 6.0 MB
None


In [13]:
# Add columns for date,month,day and year on each trips
all_trips['date'] = all_trips['started_at'].dt.date
all_trips['month'] = all_trips['started_at'].dt.month
all_trips['year'] = all_trips['started_at'].dt.year
all_trips['day'] = all_trips['started_at'].dt.day
all_trips['day_of_week'] = all_trips['started_at'].dt.day_name()

all_trips.head() #to check


,ride_id,started_at,ended_at,rideable_type,start_station_id,start_station_name,end_station_id,end_station_name,member_casual,date,month,year,day,day_of_week
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,2167,199,Wabash Ave & Grand Ave,84.0,Milwaukee Ave & Grand Ave,member,2019-01-01,1,2019,1,Tuesday
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,4386,44,State St & Randolph St,624.0,Dearborn St & Van Buren St (*),member,2019-01-01,1,2019,1,Tuesday
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,1524,15,Racine Ave & 18th St,644.0,Western Ave & Fillmore St (*),member,2019-01-01,1,2019,1,Tuesday
3,21742446,2019-01-01 00:13:45,2019-01-01 00:43:28,252,123,California Ave & Milwaukee Ave,176.0,Clark St & Elm St,member,2019-01-01,1,2019,1,Tuesday
4,21742447,2019-01-01 00:14:52,2019-01-01 00:20:56,1170,173,Mies van der Rohe Way & Chicago Ave,35.0,Streeter Dr & Grand Ave,member,2019-01-01,1,2019,1,Tuesday


In [14]:
""" (3) You will want to add a calculated field for length of ride since the 2020Q1 data did not have the "tripduration" column. We will 
add "ride_length" to the entire data frame for consistency. """

# Add a "ride_length" calculation to all_trips (in seconds)

all_trips['ride_length'] = (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds()

print(all_trips.info())
all_trips.head()

<class 'pandas.DataFrame'>
RangeIndex: 791956 entries, 0 to 791955
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             791956 non-null  str           
 1   started_at          791956 non-null  datetime64[us]
 2   ended_at            791956 non-null  datetime64[us]
 3   rideable_type       791956 non-null  str           
 4   start_station_id    791956 non-null  int64         
 5   start_station_name  791956 non-null  str           
 6   end_station_id      791955 non-null  float64       
 7   end_station_name    791955 non-null  str           
 8   member_casual       791956 non-null  str           
 9   date                791956 non-null  object        
 10  month               791956 non-null  int32         
 11  year                791956 non-null  int32         
 12  day                 791956 non-null  int32         
 13  day_of_week         791956 non-null  str

,ride_id,started_at,ended_at,rideable_type,start_station_id,start_station_name,end_station_id,end_station_name,member_casual,date,month,year,day,day_of_week,ride_length
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,2167,199,Wabash Ave & Grand Ave,84.0,Milwaukee Ave & Grand Ave,member,2019-01-01,1,2019,1,Tuesday,390.0
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,4386,44,State St & Randolph St,624.0,Dearborn St & Van Buren St (*),member,2019-01-01,1,2019,1,Tuesday,441.0
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,1524,15,Racine Ave & 18th St,644.0,Western Ave & Fillmore St (*),member,2019-01-01,1,2019,1,Tuesday,829.0
3,21742446,2019-01-01 00:13:45,2019-01-01 00:43:28,252,123,California Ave & Milwaukee Ave,176.0,Clark St & Elm St,member,2019-01-01,1,2019,1,Tuesday,1783.0
4,21742447,2019-01-01 00:14:52,2019-01-01 00:20:56,1170,173,Mies van der Rohe Way & Chicago Ave,35.0,Streeter Dr & Grand Ave,member,2019-01-01,1,2019,1,Tuesday,364.0


In [15]:
""" (4) There are some rides where tripduration shows up as negative, including several hundred rides where Divvy took bikes out
of circulation for Quality Control reasons. You will want to delete these rides."""

# Create a new version of the data frame (v2) since data is being removed

all_trips_v2 = all_trips[(all_trips['start_station_name'] != "HQ QR") & (all_trips['ride_length'] >= 0)].copy()

""" 
Adding .copy() at the very end is a crucial pandas best practice here.Without .copy(), all_trips_v2 would just be a "view" pointing back
to a slice of the original all_trips DataFrame. If you tried to modify all_trips_v2 later (like changing a value or adding a column), pandas
would throw a frustrating SettingWithCopyWarning.By using .copy(), you explicitly tell Python to allocate new memory and create a brand new,
standalone DataFrame.
"""
print(all_trips_v2.info())


<class 'pandas.DataFrame'>
Index: 788189 entries, 0 to 791955
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             788189 non-null  str           
 1   started_at          788189 non-null  datetime64[us]
 2   ended_at            788189 non-null  datetime64[us]
 3   rideable_type       788189 non-null  str           
 4   start_station_id    788189 non-null  int64         
 5   start_station_name  788189 non-null  str           
 6   end_station_id      788189 non-null  float64       
 7   end_station_name    788189 non-null  str           
 8   member_casual       788189 non-null  str           
 9   date                788189 non-null  object        
 10  month               788189 non-null  int32         
 11  year                788189 non-null  int32         
 12  day                 788189 non-null  int32         
 13  day_of_week         788189 non-null  str     

### Analysis (all figures in Seconds)

In [16]:
# Descriptive analysis on ride_length 
print(all_trips_v2['ride_length'].describe())

count    7.881890e+05
mean     1.189459e+03
std      3.329191e+04
min      1.000000e+00
25%      3.310000e+02
50%      5.390000e+02
75%      9.120000e+02
max      1.063202e+07
Name: ride_length, dtype: float64


In [17]:
""" For precise analysis. We want to know how riders user the app in a 24 hour(1day) basis. I notice that there are rider who uses the bike for more than a days,
 leaving them in will heavily distort your averages and give you a warped picture of how everyday users actually use the bike-share service.
 """
all_trips_v3 = all_trips_v2[all_trips_v2['ride_length'] <= 86400]

all_trips_v3['ride_length'].describe()

count    787707.000000
mean        823.403403
std        1866.688292
min           1.000000
25%         330.000000
50%         539.000000
75%         911.000000
max       86155.000000
Name: ride_length, dtype: float64

In [18]:
# Compare members and casual users
all_trips_v3.groupby('member_casual')['ride_length'].agg(['mean','median','max','min','count'])

,mean,median,max,min,count
member_casual,,,,,
casual,2302.732828,1388.0,86155.0,2.0,67582
member,684.571892,508.0,85984.0,1.0,720125


In [19]:
# See the average ride time by each day for members vs casual users
all_trips_v3.groupby(['member_casual','day_of_week'])['ride_length'].agg(['mean']).reset_index()

,member_casual,day_of_week,mean
0,casual,Friday,2227.490277
1,casual,Monday,2065.589154
2,casual,Saturday,2336.740906
3,casual,Sunday,2451.430132
4,casual,Thursday,2116.284588
5,casual,Tuesday,2133.807903
6,casual,Wednesday,2467.004572
7,member,Friday,666.075201
8,member,Monday,666.286590
9,member,Saturday,746.138192


In [20]:
# Analyze ridership data by type and weekday

all_trips_v3.groupby(['member_casual','day_of_week']).agg(number_of_rides = ('ride_id','count') , ave_duration = ('ride_length','mean')).reset_index()

,member_casual,day_of_week,number_of_rides,ave_duration
0,casual,Friday,7971,2227.490277
1,casual,Monday,5569,2065.589154
2,casual,Saturday,13416,2336.740906
3,casual,Sunday,18578,2451.430132
4,casual,Thursday,7105,2116.284588
5,casual,Tuesday,7288,2133.807903
6,casual,Wednesday,7655,2467.004572
7,member,Friday,115132,666.075201
8,member,Monday,110412,666.286590
9,member,Saturday,59381,746.138192


In [21]:
# finding thrends in riders how they use bike sharing app on a monthly basis for 2 yrs


df_sort = all_trips_v3.groupby(['member_casual','month','year']).agg(number_of_rides = ('ride_id','count') , ave_duration = ('ride_length','mean')).reset_index()

df_sort.sort_values(['member_casual','year']).reset_index(drop = True)

,member_casual,month,year,number_of_rides,ave_duration
0,casual,1,2019,4591,2017.229580
1,casual,2,2019,2627,1760.849638
2,casual,3,2019,15877,2205.292751
3,casual,1,2020,7721,2260.706774
4,casual,2,2020,12251,2355.210024
5,casual,3,2020,24515,2464.385437
6,member,1,2019,98601,687.313506
7,member,2,2019,93522,673.513494
8,member,3,2019,149659,675.857229
9,member,1,2020,136082,646.422231


In [23]:
# station where casual rider has high volume since we are targeting casual rider to subscribe to annual membership
#for start_station's
start_station = all_trips_v3.groupby(['member_casual','start_station_name']).agg(number_of_rider = ('ride_id','count')).reset_index()
start_station[start_station['member_casual'] == 'casual'].sort_values('number_of_rider',ascending=False).head(10)

,member_casual,start_station_name,number_of_rider
542,casual,Streeter Dr & Grand Ave,2741
325,casual,Lake Shore Dr & Monroe St,2727
482,casual,Shedd Aquarium,1831
400,casual,Millennium Park,1403
394,casual,Michigan Ave & Oak St,1017
396,casual,Michigan Ave & Washington St,835
209,casual,Dusable Harbor,829
7,casual,Adler Planetarium,826
544,casual,Theater on the Lake,794
326,casual,Lake Shore Dr & North Blvd,606


In [24]:
# for end_station's
end_station = all_trips_v3.groupby(['member_casual','end_station_name']).agg(number_of_rider = ('ride_id','count')).reset_index()
end_station[end_station['member_casual']=='casual'].sort_values('number_of_rider',ascending=False).head(10).reset_index(drop=True)

,member_casual,end_station_name,number_of_rider
0,casual,Streeter Dr & Grand Ave,3782
1,casual,Lake Shore Dr & Monroe St,2158
2,casual,Millennium Park,1934
3,casual,Shedd Aquarium,1460
4,casual,Michigan Ave & Oak St,1184
5,casual,Theater on the Lake,1073
6,casual,Michigan Ave & Washington St,960
7,casual,Lake Shore Dr & North Blvd,770
8,casual,Wabash Ave & Grand Ave,735
9,casual,Adler Planetarium,712


## EXPORT to csv file for visualization

In [ ]:
# Export to csv for visualization

for_vis=all_trips_v3.groupby(['member_casual','day_of_week'])['ride_length'].agg(['mean']).reset_index()
for_vis.to_csv('ave_duration_per_type.csv',index = False) #index = False ,This tells pandas not to write the row numbers into the final CSV file.



In [26]:
for_vis2 = df_sort.sort_values(['member_casual','year']).reset_index(drop = True)
for_vis2.to_csv('monthly_.csv',index = False)